# Statistical monitoring of a variable-speed centrifugal pump: regime-normalized residuals vs raw-signal thresholds

**Гатауллин И.С., Арьков В.Ю., УУНиТ.** Блокнот полностью воспроизводит все таблицы и рисунки препринта.

**Идея.** Обычные правила поиска выбросов из учебников статистики (правило 3σ, границы Тьюки по IQR, квантили KDE) применяют прямо к сигналам датчиков. У насоса с частотным приводом сигналы сильно меняются вместе с режимом работы (оборотами), поэтому ранние неисправности «прячутся» внутри нормального разброса. Решение: сначала убрать влияние режима линейной регрессией (`LinearRegression`), а уже потом искать выбросы в остатках.

**Что делает блокнот:**
1. `pumpsim.py` — модель насоса (законы подобия, характеристика насоса и сети, КПД, мощность, вибрация), режим работы, шум датчиков и три неисправности: износ рабочего колеса, засор, деградация подшипника; плюс 5 детекторов.
2. `run_experiments.py` — метод Монте-Карло: калибровка порогов на равную вероятность ложной тревоги (10 % за 20 дней) и сравнение детекторов на 1600 прогонах (около 3 минут).
3. `make_figures.py` — рисунки и сводные числа.

In [ ]:
%%writefile pumpsim.py
"""Physically motivated simulator of a variable-speed centrifugal pump and
statistical detectors used in the study.

All modelling assumptions are stated explicitly in the paper (Section 2).
"""
import numpy as np
from scipy.stats import gaussian_kde
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# ----------------------------------------------------------------------------
# Pump and system parameters (rated speed s = n / n_rated = 1)
# ----------------------------------------------------------------------------
P = dict(
    H0=50.0,        # shut-off head at rated speed, m
    a=1.0e-3,       # head-curve coefficient, m/(m3/h)^2
    Q_bep=100.0,    # best-efficiency flow at rated speed, m3/h
    eta_max=0.80,   # peak hydraulic efficiency
    c_eta=0.60,     # curvature of the efficiency parabola
    eta_motor=0.92, # motor + drive efficiency
    H_st=15.0,      # mean static head of the system, m
    k=2.5e-3,       # system resistance coefficient, m/(m3/h)^2
    v0=1.2,         # healthy vibration velocity RMS at BEP and rated speed, mm/s
    v_exp=1.5,      # vibration-vs-speed exponent (assumption)
    v_off=0.5,      # vibration increase away from BEP (assumption)
    rho=1000.0, g=9.81,
)
DT_MIN = 10                         # sampling period, minutes
SPD = 24 * 60 // DT_MIN             # samples per day = 144
TRAIN_DAYS, TEST_DAYS = 30, 60
SIGNALS = ["Q", "H", "P", "v"]      # monitored sensors


def operating_regime(n, rng):
    """Speed setpoint s(t) and tank level (static-head variation) for n samples."""
    t = np.arange(n) / SPD                               # time in days
    s = np.empty(n)
    i = 0
    while i < n:                                         # piecewise-constant setpoints
        dur = rng.integers(2 * 6, 8 * 6 + 1)             # 2-8 hours (10-min samples)
        s[i:i + dur] = rng.uniform(0.72, 1.0)
        i += dur
    s = np.clip(s + 0.02 * np.sin(2 * np.pi * t), 0.70, 1.0)   # small daily pattern
    level = 1.0 * np.sin(2 * np.pi * (t - 0.25)) + np.cumsum(rng.normal(0, 0.005, n))
    level -= np.mean(level)
    return t, s, level


def pump_physics(s, level, wear, clog, bearing):
    """True (noise-free) operating point for given regime and fault state."""
    H_st = P["H_st"] + level
    k = P["k"] * (1.0 + clog)
    h0 = (1.0 - wear) * P["H0"] * s ** 2
    aw = (1.0 - wear) * P["a"]
    Q = np.sqrt(np.maximum(h0 - H_st, 0.0) / (aw + k))           # pump curve = system curve
    H = H_st + k * Q ** 2
    x = (Q / s - P["Q_bep"]) / P["Q_bep"]                         # distance from BEP (affinity-scaled)
    eta = P["eta_max"] * (1.0 - 0.5 * wear) * (1.0 - P["c_eta"] * x ** 2)
    eta = np.clip(eta, 0.30, None)
    P_el = P["rho"] * P["g"] * (Q / 3600.0) * H / eta / P["eta_motor"] / 1000.0   # kW
    v = P["v0"] * s ** P["v_exp"] * (1.0 + P["v_off"] * x ** 2) * bearing
    return Q, H, P_el, v


def fault_profile(kind, t, t0, sev=1.0):
    """Fault state trajectories; zero before onset t0 (days). sev scales magnitude."""
    tau = np.clip(t - t0, 0, None)
    wear = np.zeros_like(t); clog = np.zeros_like(t); bearing = np.ones_like(t)
    if kind == "wear":        # impeller wear: head coefficient drops 6 % in 25 days
        wear = sev * 0.06 * tau / 25.0
    elif kind == "clog":      # strainer/valve clogging: resistance +40 % in 25 days
        clog = sev * 0.40 * tau / 25.0
    elif kind == "bearing":   # exponential bearing degradation: x2.5 vibration in 25 days
        beta = 0.15
        A = 1.5 / (np.exp(beta * 25.0) - 1.0)
        bearing = 1.0 + sev * A * (np.exp(beta * tau) - 1.0)
    return wear, clog, bearing


def simulate_run(kind, rng, t0=None, sev=1.0):
    """One 90-day run: 30 healthy training days + 60 test days."""
    n = (TRAIN_DAYS + TEST_DAYS) * SPD
    t, s, level = operating_regime(n, rng)
    if kind == "healthy":
        t0 = np.inf
    elif t0 is None:
        t0 = TRAIN_DAYS + rng.uniform(5, 35)
    wear, clog, bearing = fault_profile(kind, t, t0, sev)
    Q, H, P_el, v = pump_physics(s, level, wear, clog, bearing)
    meas = dict(
        s=s * (1 + rng.normal(0, 0.002, n)),
        level=level + rng.normal(0, 0.05, n),
        Q=Q * (1 + rng.normal(0, 0.005, n)),
        H=H * (1 + rng.normal(0, 0.005, n)) + rng.normal(0, 0.1, n),
        P=P_el * (1 + rng.normal(0, 0.01, n)),
        v=v * np.exp(rng.normal(0, 0.08, n)),
    )
    truth = dict(t=t, t0=t0, wear=wear, clog=clog, bearing=bearing,
                 Q=Q, H=H, P=P_el, v=v)
    return meas, truth


# ----------------------------------------------------------------------------
# Detectors. Every detector returns a per-sample score on the test part and a
# default threshold taken from the textbook rule; alarm = score > threshold.
# ----------------------------------------------------------------------------
N_TR = TRAIN_DAYS * SPD


def _split(meas):
    tr = {k: v[:N_TR] for k, v in meas.items()}
    te = {k: v[N_TR:] for k, v in meas.items()}
    return tr, te


def score_raw_3sigma(meas):
    tr, te = _split(meas)
    z = [np.abs(te[c] - tr[c].mean()) / tr[c].std() for c in SIGNALS]
    return np.max(z, axis=0), 3.0


def score_raw_iqr(meas):
    tr, te = _split(meas)
    d = []
    for c in SIGNALS:
        q1, q3 = np.percentile(tr[c], [25, 75]); iqr = q3 - q1
        d.append(np.maximum((q1 - te[c]) / iqr, (te[c] - q3) / iqr))
    return np.max(d, axis=0), 1.5          # Tukey fences


def score_raw_kde(meas, grid=512):
    """Score = -log10 of the two-sided tail probability under a KDE of training data."""
    tr, te = _split(meas)
    sc = []
    for c in SIGNALS:
        x = tr[c]
        kde = gaussian_kde(x)
        lo, hi = x.min() - 4 * x.std(), x.max() + 4 * x.std()
        g = np.linspace(lo, hi, grid)
        cdf = np.cumsum(kde(g)); cdf /= cdf[-1]
        F = np.interp(te[c], g, cdf, left=0.0, right=1.0)
        tail = np.clip(np.minimum(F, 1 - F), 1e-12, None)
        sc.append(-np.log10(tail))
    return np.max(sc, axis=0), -np.log10(0.0005)   # 0.05 % / 99.95 % percentage points


def regime_features(m):
    """Operating-regime features used as regressors (speed, flow, tank level)."""
    return np.column_stack([m["s"], m["Q"], m["level"]])


REG_MODELS = {   # target: (feature columns, polynomial degree)
    "H": ([0, 1, 2], 2),       # pump-side head:    H = f(s, Q, level)
    "P": ([0, 1, 2], 3),       # pump-side power:   P = f(s, Q, level)
    "v": ([0, 1, 2], 2),       # vibration:         v = f(s, Q, level)
    "Q": ([0, 2], 2),          # system side flow:  Q = f(s, level)
}


def residuals(meas):
    tr, te = _split(meas)
    Xtr, Xte = regime_features(tr), regime_features(te)
    out_tr, out_te = {}, {}
    for c, (cols, deg) in REG_MODELS.items():
        pf = PolynomialFeatures(deg, include_bias=False)
        A = pf.fit_transform(Xtr[:, cols]); B = pf.transform(Xte[:, cols])
        ytr = np.log(tr[c]) if c == "v" else tr[c]
        yte = np.log(te[c]) if c == "v" else te[c]
        reg = LinearRegression().fit(A, ytr)
        rtr = ytr - reg.predict(A); rte = yte - reg.predict(B)
        sd = rtr.std()
        out_tr[c], out_te[c] = rtr / sd, rte / sd
    return out_tr, out_te


def score_res_3sigma(meas, res=None):
    _, rte = res if res is not None else residuals(meas)
    return np.max([np.abs(rte[c]) for c in SIGNALS], axis=0), 3.0


def ewma(x, lam):
    """Exponentially weighted moving average z_t = lam*x_t + (1-lam)*z_{t-1}, z_0 = 0."""
    from scipy.signal import lfilter
    return lfilter([lam], [1.0, -(1.0 - lam)], x)


def score_res_ewma(meas, res=None, lam=0.1):
    _, rte = res if res is not None else residuals(meas)
    k = np.sqrt(lam / (2 - lam))            # asymptotic std of EWMA of N(0,1)
    return np.max([np.abs(ewma(rte[c], lam)) / k for c in SIGNALS], axis=0), 3.0


DETECTORS = {
    "Raw 3-sigma": score_raw_3sigma,
    "Raw Tukey IQR": score_raw_iqr,
    "Raw KDE quantile": score_raw_kde,
    "Residual 3-sigma": score_res_3sigma,
    "Residual EWMA": score_res_ewma,
}


def all_scores(meas):
    res = residuals(meas)
    out = {}
    for name, f in DETECTORS.items():
        out[name] = f(meas, res) if name.startswith("Residual") else f(meas)
    return out

In [ ]:
%%writefile run_experiments.py
"""Monte Carlo experiments: false-alarm rates and fault detection performance."""
import json, sys, time
import numpy as np
sys.path.insert(0, __file__.rsplit("/", 1)[0])
import pumpsim as ps

SEED = 2026
N_CAL, N_HEALTHY, N_FAULT, N_SWEEP = 200, 200, 200, 100
FAULTS = ["wear", "clog", "bearing"]
SWEEP = [0.5, 2.0]                   # fault-rate multipliers for the sensitivity study
TARGET_P_FA = 0.10                   # P(at least one false alarm in a 20-day window)
HORIZON = 20.0                       # days after onset used for detection probability
names = list(ps.DETECTORS)
rng = np.random.default_rng(SEED)
t_test = np.arange(ps.TEST_DAYS * ps.SPD) / ps.SPD + ps.TRAIN_DAYS   # days


def scores_of(kind, sev=1.0):
    m, tr = ps.simulate_run(kind, rng, sev=sev)
    sc = ps.all_scores(m)
    return {k: v[0].astype(np.float32) for k, v in sc.items()}, tr


t_start = time.time()
# 1) calibration runs (healthy): threshold = 90 % quantile of the maximum score
#    over non-overlapping 20-day windows -> P(false alarm in 20 days) = 10 %
W = int(HORIZON * ps.SPD)
n_win = (ps.TEST_DAYS * ps.SPD) // W


def window_max(x):
    return x[:n_win * W].reshape(n_win, W).max(axis=1)


cal = {k: [] for k in names}
for _ in range(N_CAL):
    s, _ = scores_of("healthy")
    for k in names:
        cal[k].append(window_max(s[k]))
default_thr = {"Raw 3-sigma": 3.0, "Raw Tukey IQR": 1.5, "Raw KDE quantile": float(-np.log10(0.0005)),
               "Residual 3-sigma": 3.0, "Residual EWMA": 3.0}
calib_thr = {k: float(np.quantile(np.concatenate(cal[k]), 1 - TARGET_P_FA)) for k in names}

# 2) independent healthy runs -> false alarm statistics
fa = {mode: {k: {"samples": [], "win": []} for k in names} for mode in ("default", "calibrated")}
for _ in range(N_HEALTHY):
    s, _ = scores_of("healthy")
    for k in names:
        wm = window_max(s[k])
        for mode, thr in (("default", default_thr), ("calibrated", calib_thr)):
            al = s[k] > thr[k]
            events = int(al[0]) + int((al[1:] & ~al[:-1]).sum())   # alarm episodes
            fa[mode][k]["samples"].append(events)
            fa[mode][k]["win"].extend((wm > thr[k]).tolist())


def severity_at(kind, tr, idx):
    """Physical fault magnitude at sample idx (test indexing)."""
    j = ps.N_TR + idx
    if kind == "wear":
        return 100 * tr["wear"][j]                    # % head-coefficient loss
    if kind == "clog":
        return 100 * tr["clog"][j]                    # % increase of system resistance
    return tr["bearing"][j]                           # vibration multiplier


def detect(kind, sev, n_runs):
    res = {mode: {k: {"delay": [], "sev": [], "early": 0} for k in names}
           for mode in ("default", "calibrated")}
    for _ in range(n_runs):
        s, tr = scores_of(kind, sev)
        t0 = tr["t0"]
        after = t_test >= t0
        for mode, thr in (("default", default_thr), ("calibrated", calib_thr)):
            for k in names:
                al = s[k] > thr[k]
                if al[~after].any():
                    res[mode][k]["early"] += 1          # alarm before onset (false alarm)
                hit = np.flatnonzero(al & after)
                if hit.size:
                    d = t_test[hit[0]] - t0
                    res[mode][k]["delay"].append(float(d))
                    res[mode][k]["sev"].append(float(severity_at(kind, tr, hit[0])))
                else:
                    res[mode][k]["delay"].append(float("inf"))
                    res[mode][k]["sev"].append(float("nan"))
    return res


results = {"thresholds": {"default": default_thr, "calibrated": calib_thr},
           "false_alarms": {}, "faults": {}, "sweep": {}}
for mode in fa:
    results["false_alarms"][mode] = {
        k: {"episodes_per_30_days": float(np.mean(fa[mode][k]["samples"]) / ps.TEST_DAYS * 30),
            "p_any_in_20_days": float(np.mean(fa[mode][k]["win"]))}
        for k in names}

for kind in FAULTS:
    results["faults"][kind] = detect(kind, 1.0, N_FAULT)
    for sv in SWEEP:
        results["sweep"][f"{kind}@{sv}"] = detect(kind, sv, N_SWEEP)

results["meta"] = dict(seed=SEED, n_cal=N_CAL, n_healthy=N_HEALTHY, n_fault=N_FAULT,
                       n_sweep=N_SWEEP, target_p_fa_20d=TARGET_P_FA,
                       horizon_days=HORIZON, runtime_s=round(time.time() - t_start, 1))
json.dump(results, open(sys.argv[1] if len(sys.argv) > 1 else "results.json", "w"), indent=1)
print("done in", results["meta"]["runtime_s"], "s")

In [ ]:
!python run_experiments.py results.json

In [ ]:
%%writefile make_figures.py
"""Figures (PDF) and summary numbers for the paper."""
import json, sys, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pumpsim as ps

OUT = sys.argv[1] if len(sys.argv) > 1 else "fig"
os.makedirs(OUT, exist_ok=True)
R = json.load(open(sys.argv[2] if len(sys.argv) > 2 else "results.json"))

# colour palette
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
LS = [(0, (1, 1.5)), (0, (4, 1.5)), (0, (6, 2, 1, 2)), "-", "-"]
LW = [1.6, 1.6, 1.6, 1.8, 2.4]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
NAMES = list(ps.DETECTORS)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8.5, "axes.titlesize": 9,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
    "savefig.bbox": "tight", "savefig.dpi": 300,
})
FAULT_LABEL = {"wear": "Impeller wear", "clog": "Clogging", "bearing": "Bearing degradation"}
SEV_UNIT = {"wear": "% head loss", "clog": "% resistance increase", "bearing": "vibration factor"}
summary = {}

# ---------------------------------------------------------------- Fig. 1
rng = np.random.default_rng(7)
m_h, tr_h = ps.simulate_run("healthy", rng)
m_w, tr_w = ps.simulate_run("wear", np.random.default_rng(8), t0=ps.TRAIN_DAYS + 5.0, sev=1.0)
n_tr = ps.N_TR
fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.6))
ax = axes[0]
ax.scatter(m_h["s"][:n_tr], m_h["H"][:n_tr], s=2, color=MUTED, alpha=0.35, lw=0, label="Healthy (training)")
late = (tr_w["t"] >= ps.TRAIN_DAYS + 25) & (tr_w["t"] < ps.TRAIN_DAYS + 30)
ax.scatter(m_w["s"][late], m_w["H"][late], s=4, color=C[1], alpha=0.8, lw=0,
           label=f"Worn impeller ({100*tr_w['wear'][late].mean():.1f}% head-coeff. loss)")
mu, sd = m_h["H"][:n_tr].mean(), m_h["H"][:n_tr].std()
for v in (mu - 3 * sd, mu + 3 * sd):
    ax.axhline(v, color=INK2, lw=0.9, ls="--")
ax.text(0.705, mu - 3 * sd + 0.6, "raw 3σ limits", color=INK2, fontsize=7.5)
ax.set_xlabel("Relative speed s = n / n$_{rated}$"); ax.set_ylabel("Head H, m")
ax.set_title("(a) Raw head: fault hidden in regime variation", loc="left")
ax.legend(loc="upper left", markerscale=3, fontsize=7.5)
ax = axes[1]
res_tr, res_te = ps.residuals(m_w)
t_te = tr_w["t"][n_tr:]
ax.plot(t_te - ps.TRAIN_DAYS, res_te["H"], color=MUTED, lw=0.4, alpha=0.6, label="Standardized residual of H")
ax.plot(t_te - ps.TRAIN_DAYS, ps.ewma(res_te["H"], 0.1) / np.sqrt(0.1 / 1.9), color=C[4], lw=1.8,
        label="EWMA of residual (scaled)")
ax.axvline(5.0, color=INK, lw=0.9, ls=":"); ax.text(5.4, 6.3, "fault onset", fontsize=7.5, color=INK)
thr = R["thresholds"]["calibrated"]["Residual EWMA"]
ax.axhline(-thr, color=INK2, lw=0.9, ls="--")
ax.text(29.5, -thr + 0.8, "calibrated alarm limit", ha="right", fontsize=7.5, color=INK2)
ax.set_xlabel("Test time, days"); ax.set_ylabel("Standardized units")
ax.set_xlim(0, 30); ax.set_ylim(-30, 8)
ax.set_title("(b) Regression residual reveals the drift", loc="left")
ax.legend(loc="lower left", fontsize=7.5)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_regime_masking.pdf"); fig.savefig(f"{OUT}/fig1_regime_masking.png", dpi=150); plt.close(fig)

# ---------------------------------------------------------------- Fig. 2
cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
raw = np.corrcoef([m_h[c][:n_tr] for c in ps.SIGNALS])
r_tr, _ = ps.residuals(m_h)
rr = np.corrcoef([r_tr[c] for c in ps.SIGNALS])
summary["corr_raw"] = np.round(raw, 2).tolist(); summary["corr_res"] = np.round(rr, 2).tolist()
lab = ["Q", "H", "P", "v"]
fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.4))
for ax, M, ttl in zip(axes, (raw, rr), ("(a) Raw signals", "(b) Regression residuals")):
    ax.imshow(M, cmap=cmap, vmin=-1, vmax=1); ax.grid(False)
    ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(lab); ax.set_yticklabels(lab)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7.5, color=INK)
    ax.set_title(ttl, loc="left")
    for sp in ax.spines.values(): sp.set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_correlations.pdf"); fig.savefig(f"{OUT}/fig2_correlations.png", dpi=150); plt.close(fig)

# ---------------------------------------------------------------- Fig. 3: cumulative detection
H = R["meta"]["horizon_days"]
grid = np.linspace(0, H, 201)
fig, axes = plt.subplots(1, 3, figsize=(6.8, 2.4), sharey=True)
for ax, kind in zip(axes, ["wear", "clog", "bearing"]):
    res = R["faults"][kind]["calibrated"]
    for i, k in enumerate(NAMES):
        d = np.array(res[k]["delay"])
        ax.plot(grid, [(d <= g).mean() for g in grid], color=C[i], ls=LS[i], lw=LW[i], label=k)
    ax.axhline(R["meta"]["target_p_fa_20d"], color=MUTED, lw=0.8, ls="--")
    ax.set_title(FAULT_LABEL[kind], loc="left"); ax.set_xlabel("Days since fault onset")
    ax.set_xlim(0, H); ax.set_ylim(0, 1.02)
axes[0].set_ylabel("Share of runs detected")
axes[0].text(H * 0.98, 0.14, "false-alarm level", ha="right", va="bottom", fontsize=7, color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=5, fontsize=7.5, bbox_to_anchor=(0.5, 1.08), handlelength=3)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_detection_curves.pdf"); fig.savefig(f"{OUT}/fig3_detection_curves.png", dpi=150); plt.close(fig)

# ---------------------------------------------------------------- summary numbers
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [c - h, c + h]


def summarize(res):
    out = {}
    for k in NAMES:
        d = np.array(res[k]["delay"]); s = np.array(res[k]["sev"]); w = d <= H
        out[k] = dict(p=float(w.mean()), ci=wilson(int(w.sum()), len(d)),
                      delay_med=float(np.median(d[w])) if w.any() else None,
                      delay_q1=float(np.percentile(d[w], 25)) if w.any() else None,
                      delay_q3=float(np.percentile(d[w], 75)) if w.any() else None,
                      sev_med=float(np.nanmedian(s[w])) if w.any() else None,
                      n=len(d))
    return out


summary["faults"] = {k: summarize(R["faults"][k]["calibrated"]) for k in R["faults"]}
summary["faults_default"] = {k: summarize(R["faults"][k]["default"]) for k in R["faults"]}
summary["sweep"] = {k: summarize(v["calibrated"]) for k, v in R["sweep"].items()}
summary["false_alarms"] = R["false_alarms"]; summary["thresholds"] = R["thresholds"]
summary["meta"] = R["meta"]

# R^2 of the regime regressions on training data (mean over 20 healthy runs)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
r2 = {c: [] for c in ps.REG_MODELS}
sdr = {c: [] for c in ps.REG_MODELS}
g2 = np.random.default_rng(99)
for _ in range(20):
    m, _ = ps.simulate_run("healthy", g2)
    X = ps.regime_features({k: v[:n_tr] for k, v in m.items()})
    for c, (cols, deg) in ps.REG_MODELS.items():
        A = PolynomialFeatures(deg, include_bias=False).fit_transform(X[:, cols])
        y = np.log(m[c][:n_tr]) if c == "v" else m[c][:n_tr]
        reg = LinearRegression().fit(A, y)
        r2[c].append(reg.score(A, y))
        res = y - reg.predict(A)
        sdr[c].append(res.std() / y.std())
summary["r2"] = {c: float(np.mean(v)) for c, v in r2.items()}
summary["resid_sd_ratio"] = {c: float(np.mean(v)) for c, v in sdr.items()}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
print(json.dumps({"r2": summary["r2"], "ratio": summary["resid_sd_ratio"]}, indent=1))

In [ ]:
!python make_figures.py fig results.json

In [ ]:
%%writefile make_tables.py
"""LaTeX tables generated directly from fig/summary.json."""
import json, sys, os
S = json.load(open(sys.argv[1]))
OUT = sys.argv[2]
NAMES = ["Raw 3-sigma", "Raw Tukey IQR", "Raw KDE quantile", "Residual 3-sigma", "Residual EWMA"]
TEX = {"Raw 3-sigma": r"Raw $3\sigma$", "Raw Tukey IQR": r"Raw Tukey fences",
       "Raw KDE quantile": r"Raw KDE quantiles", "Residual 3-sigma": r"Residual $3\sigma$",
       "Residual EWMA": r"Residual EWMA"}
CHANCE = {k: v["p_any_in_20_days"] for k, v in S["false_alarms"]["calibrated"].items()}


def f(x, d=2):
    return f"{x:.{d}f}"


# Table: false alarms
rows = []
for k in NAMES:
    a = S["false_alarms"]["default"][k]; b = S["false_alarms"]["calibrated"][k]
    rows.append(f"{TEX[k]} & {f(S['thresholds']['default'][k])} & {f(a['episodes_per_30_days'],1)} & "
                f"{f(a['p_any_in_20_days'])} & {f(S['thresholds']['calibrated'][k])} & "
                f"{f(b['episodes_per_30_days'])} & {f(b['p_any_in_20_days'])} \\\\")
open(f"{OUT}/tab_fa.tex", "w").write("\n".join(rows))

# Table: detection (calibrated)
SEV = {"wear": r"\% head coeff.", "clog": r"\% resistance", "bearing": r"vibration $\times$"}
LAB = {"wear": "Impeller wear", "clog": "Clogging", "bearing": "Bearing degradation"}


def det_rows(block, kinds):
    rows = []
    for kind in kinds:
        d = block[kind]
        rows.append(r"\multicolumn{5}{l}{\textit{" + LAB.get(kind.split("@")[0], kind) +
                    (f" (rate $\\times${kind.split('@')[1]})" if "@" in kind else "") + r"}} \\")
        for k in NAMES:
            v = d[k]
            lo, hi = v["ci"]
            above = lo > CHANCE[k]
            p = f"{f(v['p'])} [{f(lo)}, {f(hi)}]"
            if above and v["p"] < 0.5:
                dl, sv = r"$\dagger$", r"$\dagger$"
            elif above:
                dl = f"{f(v['delay_med'],1)} ({f(v['delay_q1'],1)}--{f(v['delay_q3'],1)})"
                sv = f(v["sev_med"], 2 if kind.startswith("bearing") else 1)
            else:
                dl, sv = "--", "--"
            rows.append(f"\\quad {TEX[k]} & {p} & {dl} & {sv} & {SEV[kind.split('@')[0]]} \\\\")
        rows.append(r"\addlinespace")
    return "\n".join(rows[:-1])


open(f"{OUT}/tab_det.tex", "w").write(det_rows(S["faults"], ["wear", "clog", "bearing"]))

# Table: sensitivity sweep, compact: P and median delay only
rows = []
for k in NAMES:
    cells = []
    for kind in ["wear", "clog", "bearing"]:
        for sv in ["0.5", "2.0"]:
            v = S["sweep"][f"{kind}@{sv}"][k]
            if v["ci"][0] > CHANCE[k] and v["p"] >= 0.5:
                cells.append(f"{f(v['p'])} / {f(v['delay_med'],1)}")
            elif v["ci"][0] > CHANCE[k]:
                cells.append(f"{f(v['p'])} / $\\dagger$")
            else:
                cells.append(f"{f(v['p'])} / --")
    rows.append(f"{TEX[k]} & " + " & ".join(cells) + r" \\")
open(f"{OUT}/tab_sweep.tex", "w").write("\n".join(rows))

# Regression quality
rows = []
for c, lab in [("Q", r"Flow $Q$ (system side)"), ("H", r"Head $H$"), ("P", r"Power $P$"), ("v", r"Vibration $\ln v$")]:
    rows.append(f"{lab} & {f(S['r2'][c],4)} & {f(S['resid_sd_ratio'][c],3)} \\\\")
open(f"{OUT}/tab_reg.tex", "w").write("\n".join(rows))
print("tables written")

In [ ]:
!python make_tables.py fig/summary.json fig && cat fig/tab_fa.tex fig/tab_det.tex fig/tab_sweep.tex fig/tab_reg.tex

## Рисунки

In [ ]:
from IPython.display import Image, display
for name in ["fig1_regime_masking", "fig2_correlations", "fig3_detection_curves"]:
    display(Image(f"fig/{name}.png"))

## Таблицы результатов

In [ ]:
import json
import pandas as pd
S = json.load(open("fig/summary.json"))
names = list(S["false_alarms"]["default"])

fa = pd.DataFrame({k: {
    "default thr": S["thresholds"]["default"][k],
    "default: episodes/30d": S["false_alarms"]["default"][k]["episodes_per_30_days"],
    "default: P(FA in 20d)": S["false_alarms"]["default"][k]["p_any_in_20_days"],
    "calibrated thr": S["thresholds"]["calibrated"][k],
    "calibrated: episodes/30d": S["false_alarms"]["calibrated"][k]["episodes_per_30_days"],
    "calibrated: P(FA in 20d)": S["false_alarms"]["calibrated"][k]["p_any_in_20_days"],
} for k in names}).T.round(2)
print("Ложные тревоги на 200 исправных прогонах"); display(fa)

rows = []
for kind, d in S["faults"].items():
    for k in names:
        v = d[k]
        rows.append({"fault": kind, "detector": k, "P(detect ≤20 d)": v["p"],
                     "CI low": v["ci"][0], "CI high": v["ci"][1],
                     "median delay, d": v["delay_med"], "magnitude at detection": v["sev_med"]})
det = pd.DataFrame(rows).round(3)
print("Обнаружение при откалиброванных порогах (200 прогонов на неисправность)"); display(det)

sw = []
for key, d in S["sweep"].items():
    for k in names:
        sw.append({"fault@rate": key, "detector": k, "P": d[k]["p"], "median delay, d": d[k]["delay_med"]})
print("Чувствительность к скорости развития неисправности (100 прогонов)"); display(pd.DataFrame(sw).round(3))

dd = pd.DataFrame({kind: {k: S["faults_default"][kind][k]["p"] for k in names} for kind in S["faults_default"]})
print("Доля «обнаружений» при порогах по умолчанию (с учётом случайных тревог)"); display(dd.round(3))

print("R² регрессий и отношение СКО остатков к СКО сырого сигнала")
display(pd.DataFrame({"R2": S["r2"], "resid SD / raw SD": S["resid_sd_ratio"]}).round(4))